# AML - SAIGE-GENE SET-BASED ANALYSIS - FIL NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [3]:
# Load packages 
import os

In [5]:
# Set environment variable for workspace bucket 
my_bucket = 'gs://rw-migration-aou-rw-9f879fc4'
# Set environment variable for workspace bucket 
my_bucket

%env WORKSPACE_BUCKET=gs://rw-migration-aou-rw-9f879fc4
%env ARTIFACT_REGISTRY_DOCKER_REPO=us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod

%run /home/jupyter/workspace/rw-migration-aou-rw-9f879fc4/notebooks/Setting_Env_Variables.ipynb

env: WORKSPACE_BUCKET=gs://rw-migration-aou-rw-9f879fc4
env: ARTIFACT_REGISTRY_DOCKER_REPO=us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod


# COMMAND 

In [5]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_covariates; do 
    
        for cov in age_squared age_sex age_biospecimen biospecimen; do 

            echo "Processing population: ${pop} | phenotype: ${pheno} | covariate: ${cov}"

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVarRatio=TRUE \
                --IsOverwriteVarianceRatioFile=TRUE \
                --isCateVarianceRatio=TRUE \
                --phenoCol=aml_phenotype \
                --traitType=binary \
                --covarColList="age,sex,${cov},PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10" \
                --qCovarColList=sex \
                --sampleIDColinphenoFile=person_id \
                --outputPrefix="${output}/${pop}_${pheno}_${cov}_null_glmm" 

        done 
    done
done


Overwriting saige_gene_step01.sh


In [6]:
# Save script to workspace bucket 
!gcloud storage cp saige_gene_step01.sh {my_bucket}/data/scripts/

Copying file://saige_gene_step01.sh to gs://rw-migration-aou-rw-9f879fc4/data/scripts/saige_gene_step01.sh
  Completed files 1/1 | 1.4kiB/1.4kiB                                          


In [7]:
# Check script in workspace bucket 
!gcloud storage cat {my_bucket}/data/scripts/saige_gene_step01.sh 

#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"

    for pheno in aml_covariates; do 

        for cov in age_squared age_sex age_biospecimen biospecimen; do 

            echo "Processing population: ${pop} | phenotype: ${pheno} | covariate: ${cov}"

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVa

# SUBMIT COMMAND 

In [9]:
%%bash --out saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --preemptible \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/saige_gene_step01.sh"


httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: saige-gene--vienose--260909-220936-80
  job-name: saige-gene-step01
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/saige-gene--vienose--260909-220936-80-0-0
Launched job-id: saige-gene--vienose--260909-220936-80
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--260909-220936-80' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--260909-220936-80' --users 'vienose'


In [1]:
!dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--260909-220936-80' --users 'vienose' --status '*' --full

- create-time: '2026-09-09 22:09:37.243650'
  dsub-version: v0-5-5
  end-time: '2026-09-09 22:41:30.692854'
  envs:
    grm: gs://rw-migration-aou-rw-9f879fc4/data/saige_step00/grm
    output: gs://rw-migration-aou-rw-9f879fc4/data/saige_step01/null_glmm
    phenotype: gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files
    variance_files: gs://rw-migration-aou-rw-9f879fc4/data/saige_step01/variance_files
  events:
  - event-time: '2026-09-09T22:09:47.593030042Z'
    name: scheduled
  - event-time: '2026-09-09T22:12:12.818086558Z'
    name: start
  - event-time: '2026-09-09T22:41:30.692854117Z'
    name: ok
  input-recursives:
    grm: gs://rw-migration-aou-rw-9f879fc4/data/saige_step00/grm/
    phenotype: gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/
    variance_files: gs://rw-migration-aou-rw-9f879fc4/data/saige_step01/variance_files/
  inputs: {}
  internal-id: projects/wb-fast-radish-6415/locations/us-central1/jobs/saige-gene--vienose--260909-220936-80-0-0